<!-- nav -->
[← 8.08 · C Preprocessor](08_08_C_Preprocessor.ipynb) · [Home](../../README.md) · [8.10 · Cpp RegExp →](08_10_Cpp_RegExp.ipynb)

# 8.09 · Cpp Types

C++ writes types in more places than C: in a cast, and in `static_cast<int>(x)`, `new int`, `sizeof(int)`, a
conversion function's name, `operator int`, a lambda's trailing return type, a template parameter's default. What it
writes there is a **type-id** (ISO/IEC 14882, [dcl.name]), C's type name grown with C++'s own specifiers and
declarators: qualified names, `std::string`, references, `&`, scoped enumerations, `decltype`. `Cpp Types` models
type-ids as a dialect of its own, beside [8.07 · C Types](08_07_C_Types.ipynb), since a C++ type-id holds what no C type
name can. This page builds type-ids from `int` up, then an enumeration with a fixed type; then every place C++ puts one,
what validation says, how C++ reads one in a cast, and the editions it covers.

Like the rest of Part I, it starts from `dialects()`. `S` is its holder namespace and `amd64` the ABI a C family's
holder names (tutorial 2); `identifier` writes a C++ name of one identifier (tutorial 4), `x` the C++ term that reads
`x`, and `keyword` a type specifier that is a keyword, each a spec the builders call:

In [1]:
import assert from "node:assert/strict";
import * as Cpp from "@mbse/expressions/Cpp/Expressions";
import * as CppTypes from "@mbse/expressions/Cpp/Types";
import { Standards, Symbolics, Terms } from "@mbse/expressions/Framework";
import { dialects } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const amd64 = store.Instances.Expressions.SystemV.AMD64;
const identifier = (word: string) => (n: any) => n.identifier((i: any) => i.spelling(word));
const x = (t: any) => t.idexpression((r: any) => r.name(identifier("x")));
const keyword = (word: string) => (s: any) => s.primitivetypespecifier((p: any) => p.keyword(word));
const DIALECT = CppTypes.Expressions.DIALECT;

/** What `evaluate` says of `expression`: its value, or the reason it refuses it. */
function refused(simplify: (expression: any) => unknown, expression: any): string {
  try {
    const result: any = simplify(expression);
    return String(typeof result?.evaluate === "function" ? result.evaluate() : result);
  } catch (error) {
    return (error as Error).message;
  }
}

<a id="T08.09.1"></a>
## 08.09.1 · A type-id, from `int` to `const std::string&`

```text
int        unsigned long        const std::string&
```

`int` is a `typeid` of `Cpp Types`, as in `C Types`, its specifiers and declarator helpers; but they are C++'s helpers,
which C++'s own expressions hold too (a lambda's declarator, a `new` expression's), so `Cpp Types`' `typeid` is a
schema of its own, not C's. `const std::string&` shows two of them: a specifier whose name is a `qualifiedname`, `std`
then `string`, and a `referencedeclarator`. Its holder, `S.Cpp.Types.Expression`, names its ABI, as every C family
holder does:

In [2]:
const int_ = (t: any) => t.typeid((i: any) => i.specifiers([keyword("int")]));
const type_ids: any[] = store.transaction(() => [
  int_,
  (t: any) => t.typeid((i: any) => i.specifiers([keyword("unsigned"), keyword("long")])),
  (t: any) => t.typeid((i: any) => i.specifiers([
    (s: any) => s.cvqualifier((q: any) => q.keyword("const")),
    (s: any) => s.namedtypespecifier((n: any) => n.name((m: any) => m.qualifiedname(
      (q: any) => q.qualifiers([identifier("std")]).name(identifier("string")))))])
    .declarator((d: any) => d.referencedeclarator((r: any) => r))),
].map((spec) => S.Cpp.Types.Expression((e: any) => e.root(spec).abi([amd64]))));
for (const type_id of type_ids) {
  const form = Terms.form(Terms.term(type_id));
  const declarator = form.attributes.get("declarator") as Terms.Part | undefined;
  console.log(CppTypes.Text.ToText(type_id).padEnd(19), form.kind, "of", Terms.dialectOf(Terms.term(type_id)), "| specifiers:",
    (form.attributes.get("specifiers") as Terms.Part[]).map((s) => s.kind).join(", "), "| declarator:", declarator ? declarator.kind : "-");
}
console.log(DIALECT.name(), DIALECT.tag, "| its kinds:", [...DIALECT.kinds().keys()].join(", "));
const same = store.registered("Expressions.Cpp.Types.TypeId") === store.registered("Expressions.C.Types.TypeId");
console.log("its typeid the same schema as C Types':", same ? "yes" : "no");
assert.deepEqual(type_ids.map((t) => CppTypes.Text.ToText(t)), ["int", "unsigned long", "const std::string&"]);

int                 typeid of Expressions.Cpp.Types.Term | specifiers: primitivetypespecifier | declarator: -


unsigned long       typeid of Expressions.Cpp.Types.Term | specifiers: primitivetypespecifier, primitivetypespecifier | declarator: -


const std::string&  typeid of Expressions.Cpp.Types.Term | specifiers: cvqualifier, namedtypespecifier | declarator: referencedeclarator


Cpp Types cpp_types | its kinds: typeid, enumerator, error, missing


its typeid the same schema as C Types': no


<a id="T08.09.2"></a>
## 08.09.2 · A scoped enumeration, with its fixed type

```text
enum class Color : unsigned char {
    RED = 1,
    GREEN
}
```

C++'s enumerations may be scoped, `enum class`, and fix their underlying type, `: unsigned char`. The fixed type is a
type-id, a `Cpp Types` term the specifier holds as its `base`; each enumerator is `Cpp Types`' `enumerator`, its value
C++'s term. A form gives the specifier's terms in order, its base, then its enumerators:

In [3]:
const enumerator = (word: string, value?: unknown) => (t: any) => t.enumerator(
  (r: any) => value === undefined ? r.name((i: any) => i.spelling(word)) : r.name((i: any) => i.spelling(word)).value(value));
const one = (t: any) => t.literal((r: any) => r.value((v: any) => v.as_scalar(1n)));
const color = store.transaction(() => S.Cpp.Types.Expression((e: any) => e.root((u: any) => u.typeid((i: any) => i.specifiers([
  (s: any) => s.enumspecifier((n: any) => n.key("enum class").name(identifier("Color"))
    .base((t: any) => t.typeid((i: any) => i.specifiers([keyword("unsigned"), keyword("char")])))
    .body(true).enumerators([enumerator("RED", one), enumerator("GREEN")]))]))).abi([amd64])));
console.log(CppTypes.Text.ToText(color));
for (const t of Terms.form(Terms.term(color)).arguments) {
  console.log(" ", Terms.kindOf(t)[0].padEnd(10), `of ${DIALECT.owner(t).name().padEnd(9)}`, CppTypes.Text.ToText(t));
}
assert.equal(CppTypes.Text.ToText(color), "enum class Color : unsigned char {\n    RED = 1,\n    GREEN\n}");

enum class Color : unsigned char {
    RED = 1,
    GREEN
}


  typeid     of Cpp Types unsigned char


  enumerator of Cpp Types RED = 1


  enumerator of Cpp Types GREEN


<a id="T08.09.3"></a>
## 08.09.3 · Wherever C++ puts a type

```text
(int)x        static_cast<int>(x)        new int        sizeof(int)        operator int        []<typename T = int> {}
```

C++'s kinds hold a `Cpp Types` term wherever its grammar puts a type-id, eight properties, read from the store as
tutorial 8 read them (08.3); and `typeid` stands among C++'s own kinds where an expression or a type may, in `sizeof`.
Each holder below is C++'s, and each prints the type where C++ writes it:

In [4]:
const union = store.registered(DIALECT.union);
const held = ["CastExpression", "NamedCastExpression", "NewExpression", "ConversionName", "LambdaDeclarator",
  "FunctionDeclarator", "EnumSpecifier", "TypeParameter"].flatMap((name) =>
  [...store.registered(`Expressions.Cpp.Expressions.${name}`).properties.entries()]
    .filter(([, prop]: [string, any]) => prop.schema === union).map(([p]: [string, any]) => `${name}.${p}`));
console.log(held.join(", "));
const positions: any[] = store.transaction(() => [
  (u: any) => u.castexpression((r: any) => r.type(int_).operand(x)),
  (u: any) => u.namedcastexpression((n: any) => n.operator("static_cast").type(int_).operand(x)),
  (u: any) => u.newexpression((n: any) => n.type(int_)),
  (u: any) => u.sizeofexpression((s: any) => s.operand(int_)),
  (u: any) => u.idexpression((r: any) => r.name((n: any) => n.conversionname((c: any) => c.type(int_)))),
  (u: any) => u.lambdaexpression((r: any) => r.template_parameters([(t: any) => t.typeparameter(
    (p: any) => p.key("typename").name((i: any) => i.spelling("T")).default(int_))])),
].map((spec) => S.Cpp.Expressions.Expression((e: any) => e.root(spec).abi([amd64]))));
for (const expression of positions) {
  console.log(Cpp.Text.ToText(expression).padEnd(24), Terms.kindOf(Terms.term(expression))[0]);
}
assert.deepEqual(positions.map((p) => Cpp.Text.ToText(p)), [
  "(int)x", "static_cast<int>(x)", "new int", "sizeof(int)", "operator int", "[]<typename T = int> {}"]);

CastExpression.type, NamedCastExpression.type, NewExpression.type, ConversionName.type, LambdaDeclarator.trailing_return, FunctionDeclarator.trailing_return, EnumSpecifier.base, TypeParameter.default


(int)x                   castexpression


static_cast<int>(x)      namedcastexpression


new int                  newexpression


sizeof(int)              sizeofexpression


operator int             idexpression


[]<typename T = int> {}  lambdaexpression


<a id="T08.09.4"></a>
## 08.09.4 · Checked, and what a lambda binds in its type

Validation checks a type-id as C++'s grammar has it, and C++'s kinds check that a type stands where they need one: an
enumerator's name is an identifier, and a `new` expression needs a type. A type-id binds no name of its own, but it may
read one: `decltype(a)` reads `a`. In a lambda's trailing return type, `[](int a) -> decltype(a) {}`, `a` is the
lambda's parameter, which the lambda binds (tutorial 7), so the lambda needs nothing; without the parameter, `a` is
unbound, and validation says where:

In [5]:
const decltype_a = (t: any) => t.typeid((i: any) => i.specifiers([(s: any) => s.decltypespecifier(
  (d: any) => d.expression((e: any) => e.idexpression((r: any) => r.name(identifier("a")))))]));
const parameter = (p: any) => p.parameterdeclaration((q: any) => q.specifiers([keyword("int")])
  .declarator((e: any) => e.iddeclarator((i: any) => i.name(identifier("a")))));
const checked: Record<string, any> = store.transaction(() => ({
  "a b": S.Cpp.Types.Expression((e: any) => e.root(enumerator("a b")).abi([amd64])),
  "new": S.Cpp.Expressions.Expression((e: any) => e.root((u: any) => u.newexpression((n: any) => n)).abi([amd64])),
  "[](int a) -> decltype(a) {}": S.Cpp.Expressions.Expression((e: any) => e.root((u: any) => u.lambdaexpression((r: any) => r.declarator(
    (d: any) => d.parameters([parameter]).trailing_return(decltype_a)))).abi([amd64])),
  "[] -> decltype(a) {}": S.Cpp.Expressions.Expression((e: any) => e.root((u: any) => u.lambdaexpression((r: any) => r.declarator(
    (d: any) => d.trailing_return(decltype_a)))).abi([amd64])),
}));
for (const [source, expression] of Object.entries(checked)) {
  const dialect = Terms.dialectOf(Terms.term(expression)) === DIALECT.union ? DIALECT : Cpp.Expressions.DIALECT;
  console.log(source.padEnd(27), "|", dialect.validate(expression).join("; ") || "valid",
    "| needs:", [...Symbolics.free(expression, dialect)].sort().join(", ") || "nothing");
}
assert.deepEqual(Cpp.Expressions.DIALECT.validate(checked["[](int a) -> decltype(a) {}"]), []);
assert.equal(Cpp.Text.ToText(checked["[](int a) -> decltype(a) {}"]), "[](int a) -> decltype(a) {}");

a b                         | 'a b' is not an identifier | needs: nothing


new                         | a newexpression needs a type | needs: nothing


[](int a) -> decltype(a) {} | valid | needs: nothing


[] -> decltype(a) {}        | declarator trailing_return: specifiers 0 expression: idexpression 'a' is not bound | needs: a


<a id="T08.09.5"></a>
## 08.09.5 · Read by C++, in a cast

```text
(uint8_t)(x + 1)        (signed char)200        static_cast<int>(1)
```

C++ reads `Cpp Types`' terms through the dialects it reads (tutorial 8, 08.1), so it validates, infers and simplifies a
cast with its type in it, as C does ([8.07 · C Types](08_07_C_Types.ipynb), 08.07.5): with `x` 255, `(uint8_t)(x + 1)`
folds its operand to `256` and keeps the cast, since C++ writes no literal of a `uint8_t`; `(signed char)200` stays as
written for the same reason (where it is used, the ABI's `signed char`, holding -128 to 127, makes it -56); and C++
infers the cast's domain from the type it names. C++'s rules fold the kinds C++ shares with C, and leave the others in
the residual, named: a named cast, `static_cast<int>(1)`, and a type-id itself:

In [6]:
const uint8 = (t: any) => t.typeid((i: any) => i.specifiers([(s: any) => s.namedtypespecifier((n: any) => n.name(identifier("uint8_t")))]));
const casts: any[] = store.transaction(() => [
  (u: any) => u.castexpression((r: any) => r.type(uint8).operand(
    (o: any) => o.binaryexpression((b: any) => b.left(x).operator("+").right(one)))),
  (u: any) => u.castexpression((r: any) => r.type((t: any) => t.typeid((i: any) => i.specifiers([keyword("signed"), keyword("char")])))
    .operand((o: any) => o.literal((l: any) => l.value((v: any) => v.as_scalar(200n))))),
  (u: any) => u.namedcastexpression((n: any) => n.operator("static_cast").type(int_).operand(one)),
].map((spec) => S.Cpp.Expressions.Expression((e: any) => e.root(spec).abi([amd64]))));
/** What simplifying `expression` in `scope` leaves, as C++ writes it: a literal, or the term with what it decides folded. */
function residual(expression: unknown, scope: Record<string, unknown>): string {
  const left = store.transaction(() =>
    S.Cpp.Expressions.Expression((e: any) => e.root(Cpp.Simplifiers.OfAny(expression, scope).residual).abi([amd64])));
  return Cpp.Text.ToText(left);
}
for (const expression of casts) {
  const type = Terms.form(Terms.term(expression)).arguments[0]; // each one's first subterm, its type
  console.log(Cpp.Text.ToText(expression).padEnd(20), "its type:", Terms.kindOf(type)[0], "of",
    Cpp.Expressions.DIALECT.owner(type).name(), "| x = 255:", residual(expression, { x: 255n }));
}
console.log("C++ infers", Cpp.Text.ToText(casts[1]), "as", Cpp.Expressions.DIALECT.infer(casts[1]).name());
console.log(refused((t) => Cpp.Simplifiers.OfAny(t).evaluate(), type_ids[0]));
assert.deepEqual(casts.map((c) => residual(c, { x: 255n })), ["(uint8_t)256", "(signed char)200", "static_cast<int>(1)"]);

(uint8_t)(x + 1)     its type: typeid of Cpp Types | x = 255: 0


(signed char)200     its type: typeid of Cpp Types | x = 255: -56


static_cast<int>(1)  its type: typeid of Cpp Types | x = 255: Cpp namedcastexpression is not evaluated


C++ infers (signed char)200 as signed char


Cpp Types typeid is not evaluated


<a id="T08.09.6"></a>
## 08.09.6 · Its editions

`Cpp Types` covers C++'s three editions, as C++ does, each by its designation and its common name, both of its kinds in
each ([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):

In [7]:
const types = store.Instances.Expressions.Cpp.Types; // the dialect, by its global name
for (const [designation, edition] of Standards.standards(types)) {
  console.log(designation, "|", Standards.tags(edition).slice(1).join(", "), "|",
    Standards.kinds(edition).map((kind: any) => kind.name).join(", "));
}
assert.ok(Standards.find(types, "C++20") === Standards.standards(types).get("ISO/IEC 14882:2020"));

ISO/IEC 14882:2017 | C++17 | Expressions.Cpp.Types.TypeId, Expressions.Cpp.Types.Enumerator


ISO/IEC 14882:2020 | C++20 | Expressions.Cpp.Types.TypeId, Expressions.Cpp.Types.Enumerator


ISO/IEC 14882:2024 | C++23 | Expressions.Cpp.Types.TypeId, Expressions.Cpp.Types.Enumerator


## What you learned

- **A type-id is a `Cpp Types` term**, `typeid`, of C++'s own specifiers and declarators: qualified names, references,
  `decltype`, scoped enumerations with a fixed type.
- **C++ holds one wherever its grammar puts a type**: a cast's, a named cast's, `new`'s, a conversion function's, a
  lambda's trailing return type, an enumeration's fixed type, a template parameter's default; and `sizeof(int)`.
- **It is checked as C++'s grammar has it**, and the names it reads are its scope's, a lambda's parameters among them.
- **C++ reads it in a cast**, converting and inferring by it, while a type-id itself is never folded.

Next, the regular expressions C++'s library reads, [8.10 · Cpp RegExp](08_10_Cpp_RegExp.ipynb).

<!-- nav -->
[← 8.08 · C Preprocessor](08_08_C_Preprocessor.ipynb) · [Home](../../README.md) · [8.10 · Cpp RegExp →](08_10_Cpp_RegExp.ipynb)